In [1]:
import pandas as pd
import numpy as np

def log(s):
    print(s, flush=True)

# ============================================================
# 1. Load datasets
# ============================================================
log("Loading datasets...")

az_train = pd.read_csv("AndroZoo Train Data.csv")   # 2014–2017
az_test  = pd.read_csv("AndroZoo Test Data.csv")    # 2018–2023
drebin   = pd.read_csv("Malware DREBIN.csv")

# DREBIN → label = 1로 고정
drebin["label"] = 1
log("[OK] DREBIN label set → 1")

# ============================================================
# 2. Training dataset construction (2안)
# ============================================================

log("\n===== BUILDING TRAINING DATASET (2안) =====")

# (A) AndroZoo benign 20,000
benign_20k = az_train[az_train["label"] == 0].sample(20000, random_state=42)

# (B) AndroZoo malicious 20,000
mal_20k = az_train[az_train["label"] == 1].sample(20000, random_state=42)

# (C) Extra 2,780 benign → 695 per year (2014~2017)
log("Sampling 695 benign per year (2014–2017)...")

extra_benign_list = []
for y in [2014, 2015, 2016, 2017]:
    sub = az_train[(az_train["year"] == y) & (az_train["label"] == 0)]
    extra_benign_list.append(sub.sample(695, random_state=42))

extra_benign = pd.concat(extra_benign_list, ignore_index=True)

# (D) DREBIN malicious 2,780
drebin_2780 = drebin.sample(2780, random_state=42)

# build final training dataset
train_df = pd.concat(
    [benign_20k, extra_benign, mal_20k, drebin_2780],
    ignore_index=True
)

train_df.to_csv("final_train_data.csv", index=False)

log("===== TRAINING DATASET COMPLETE =====")
log(train_df["label"].value_counts())
log(f"Total train samples: {len(train_df)}")


# ============================================================
# 3. Test Dataset 1 (1:1, benign 18k + malicious 18k)
# ============================================================

log("\n===== BUILDING TEST DATASET 1 (1:1) =====")

# (A) 18,000 benign from AndroZoo Test (2018–23)
benign_test_18k = az_test[az_test["label"] == 0].sample(18000, random_state=42)

# (B) malicious = Drebin 2,780 + AndroZoo 15,220
drebin_mal_test = drebin.drop(drebin_2780.index)   # train에서 사용되지 않은 drebin
drebin_mal_test_2780 = drebin_mal_test.sample(2780, random_state=42)

# AndroZoo malicious 15,220 (2018–2023)
az_mal_test = az_test[az_test["label"] == 1].sample(15220, random_state=42)

mal_test_1 = pd.concat([drebin_mal_test_2780, az_mal_test], ignore_index=True)

test1_df = pd.concat([benign_test_18k, mal_test_1], ignore_index=True)
test1_df.to_csv("final_test_data_1.csv", index=False)

log("===== TEST DATASET 1 COMPLETE =====")
log(test1_df["label"].value_counts())
log(f"Total test1 samples: {len(test1_df)}")


# ============================================================
# 4. Test Dataset 2 (9:1) benign 18k, malicious 2k (DREBIN only)
# ============================================================

log("\n===== BUILDING TEST DATASET 2 (9:1, DREBIN malicious) =====")

mal_test2 = drebin_mal_test.sample(2000, random_state=42)   # 2000 malicious from remaining drebin

test2_df = pd.concat([benign_test_18k, mal_test2], ignore_index=True)
test2_df.to_csv("final_test_data_2.csv", index=False)

log("===== TEST DATASET 2 COMPLETE =====")
log(test2_df["label"].value_counts())
log(f"Total test2 samples: {len(test2_df)}")


# ============================================================
# 5. Test Dataset 3 (9:1) benign 18k, malicious 2k (AndroZoo only)
# ============================================================

log("\n===== BUILDING TEST DATASET 3 (9:1, AndroZoo malicious) =====")

# AndroZoo malicious 2018–2023 → 2000개 (연도별 333~334)
az_mal_2k_list = []
years = [2018, 2019, 2020, 2021, 2022, 2023]

for y in years:
    sub = az_test[(az_test["year"] == y) & (az_test["label"] == 1)]
    # distribute approx 333–334 each
    n = 333 if y != 2023 else 334  # sum = 2000
    az_mal_2k_list.append(sub.sample(n, random_state=42))

mal_test3 = pd.concat(az_mal_2k_list, ignore_index=True)

test3_df = pd.concat([benign_test_18k, mal_test3], ignore_index=True)
test3_df.to_csv("final_test_data_3.csv", index=False)

log("===== TEST DATASET 3 COMPLETE =====")
log(test3_df["label"].value_counts())
log(f"Total test3 samples: {len(test3_df)}")

log("\n===== ALL DATASETS SUCCESSFULLY GENERATED =====")


Loading datasets...


/tmp/ipykernel_1790726/3760590665.py:13: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  az_test  = pd.read_csv("AndroZoo Test Data.csv")    # 2018–2023


[OK] DREBIN label set → 1

===== BUILDING TRAINING DATASET (2안) =====
Sampling 695 benign per year (2014–2017)...
===== TRAINING DATASET COMPLETE =====
label
0    22780
1    22780
Name: count, dtype: int64
Total train samples: 45560

===== BUILDING TEST DATASET 1 (1:1) =====
===== TEST DATASET 1 COMPLETE =====
label
0    18000
1    18000
Name: count, dtype: int64
Total test1 samples: 36000

===== BUILDING TEST DATASET 2 (9:1, DREBIN malicious) =====
===== TEST DATASET 2 COMPLETE =====
label
0    18000
1     2000
Name: count, dtype: int64
Total test2 samples: 20000

===== BUILDING TEST DATASET 3 (9:1, AndroZoo malicious) =====
===== TEST DATASET 3 COMPLETE =====
label
0    18000
1     1999
Name: count, dtype: int64
Total test3 samples: 19999

===== ALL DATASETS SUCCESSFULLY GENERATED =====


In [ ]:
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# Settings
# ============================================================
cluster_num = 19
features_to_remove = ["apkname", "label", "year"]
test_years = list(range(2018, 2024))
alpha = 0.5

def log(s):
    print(s, flush=True)

# ============================================================
# Fit GMM + Scaler
# ============================================================
def fit_gmm_and_scaler(train_df):
    log("Extracting GMM training data (2014–2017)...")
    gmm_train_df = train_df[train_df["year"].between(2014, 2017)].copy()
    X_gmm = gmm_train_df.drop(columns=features_to_remove, errors="ignore").select_dtypes(exclude=["object"])

    log("Scaling features…")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_gmm)

    log(f"Training GMM K={cluster_num} (diag covariance)…")
    gmm = GaussianMixture(n_components=cluster_num, covariance_type="diag", random_state=42)
    gmm.fit(X_scaled)

    gmm_train_df["cluster"] = gmm.predict(X_scaled)
    return gmm, scaler, gmm_train_df, X_scaled


# ============================================================
# Cluster pruning
# ============================================================
def prune_clusters(gmm_train_df, gmm, X_scaled, min_samples=20):
    log("\n=== Cluster Pruning ===")
    clusters = gmm_train_df["cluster"].values
    df_cnt = gmm_train_df.groupby(["cluster","label"]).size().unstack(fill_value=0)
    df_cnt["Total"] = df_cnt.sum(axis=1)

    deleted, alive = [], []
    for k in range(gmm.n_components):
        if k not in df_cnt.index:
            deleted.append(k)
            continue
        b = df_cnt.loc[k,0] if 0 in df_cnt.columns else 0
        m = df_cnt.loc[k,1] if 1 in df_cnt.columns else 0
        tot = b+m
        if tot < min_samples or b==0 or m==0:
            deleted.append(k)
        else:
            alive.append(k)

    log(f"Deleted clusters: {deleted}")
    log(f"Alive clusters:   {alive}")

    # Reassign deleted cluster samples
    R = gmm.predict_proba(X_scaled)
    alive = np.array(alive)
    new_clusters = clusters.copy()

    for i in range(len(new_clusters)):
        if new_clusters[i] in deleted:
            probs = R[i,alive]
            new_clusters[i] = alive[np.argmax(probs)]

    gmm_train_df["cluster"] = new_clusters
    return gmm_train_df, alive


# ============================================================
# Train RF per cluster
# ============================================================
def train_rf_models(gmm_train_df):
    log("\nTraining RF models per cluster…")
    rf_models = [None]*cluster_num

    for k in range(cluster_num):
        cluster_data = gmm_train_df[gmm_train_df["cluster"]==k]
        if len(cluster_data) < 3:
            continue

        X = cluster_data.drop(columns=["cluster","label","year","apkname"], errors="ignore").select_dtypes(exclude=["object"])
        y = cluster_data["label"]

        try:
            if y.value_counts().min() < 2:
                X_train,_,Y_train,_ = train_test_split(X,y,random_state=42)
            else:
                X_train,_,Y_train,_ = train_test_split(X,y,stratify=y,random_state=42)
        except:
            continue

        model = RandomForestClassifier(random_state=42)
        model.fit(X_train, Y_train)
        rf_models[k] = model

    return rf_models


# ============================================================
# Adaptive Thresholds
# ============================================================
def compute_thresholds(gmm_train_df):
    lbl = gmm_train_df.groupby(["cluster","label"]).size().unstack(fill_value=0)
    lbl.columns = ["Benign","Malicious"]
    lbl["Total"] = lbl.sum(axis=1)
    lbl["MR"] = lbl["Malicious"]/lbl["Total"]

    r = lbl["MR"].reindex(range(cluster_num)).fillna(0.5).values
    d = np.sqrt((2*alpha-1)**2 + (2*r-1)**2)
    u = d/np.sqrt(2)
    th = 0.5 + 0.5*np.sqrt(u)
    th = np.clip(th, 0.5, 0.98)
    return th


# ============================================================
# Mapping test data to clusters
# ============================================================
def map_yearwise(test_df, scaler, gmm):
    out = {}
    for y in test_years:
        df = test_df[test_df["year"]==y].copy()
        if df.empty:
            continue
        X = df.drop(columns=features_to_remove, errors="ignore").select_dtypes(exclude=["object"])
        Xs = scaler.transform(X)
        df["cluster"] = gmm.predict(Xs)
        out[y] = df
    return out


# ============================================================
# Evaluate
# ============================================================
def evaluate(mapped, rf_models, thresholds):
    total_p, total_t, total_prob = [], [], []
    yearly = []

    for year, df in sorted(mapped.items()):
        yp, yt, yprob = [], [], []

        for k in np.unique(df["cluster"]):
            sub = df[df["cluster"]==k]
            if sub.empty or rf_models[k] is None:
                continue

            features = rf_models[k].feature_names_in_
            X_valid = sub[features]
            y_valid = sub["label"]

            prob = rf_models[k].predict_proba(X_valid)[:,1]
            pred = (prob >= thresholds[k]).astype(int)

            yp.extend(pred)
            yt.extend(y_valid)
            yprob.extend(prob)

            total_p.extend(pred)
            total_t.extend(y_valid)
            total_prob.extend(prob)

        if yt:
            acc = accuracy_score(yt,yp)
            f1 = f1_score(yt,yp)
            try:
                auc = roc_auc_score(yt,yprob)
            except:
                auc = float("nan")
            tn,fp,fn,tp = confusion_matrix(yt,yp).ravel()
            fpr = fp/(fp+tn)
            yearly.append((year,acc,f1,auc,fpr))

    # Overall
    tn,fp,fn,tp = confusion_matrix(total_t, total_p).ravel()
    overall = (
        accuracy_score(total_t,total_p),
        f1_score(total_t,total_p),
        roc_auc_score(total_t,total_prob),
        fp/(fp+tn),
        (tn,fp,fn,tp)
    )

    return overall, pd.DataFrame(yearly, columns=["Year","ACC","F1","AUC","FPR"])


# ============================================================
# Master runner
# ============================================================
def run_experiment(test_csv, name):
    log(f"\n\n================= {name} : START =================")
    test_df = pd.read_csv(test_csv)

    mapped = map_yearwise(test_df, scaler, gmm)
    overall, yearly = evaluate(mapped, rf_models, thresholds)

    acc,f1,auc,fpr,cm = overall
    tn,fp,fn,tp = cm

    log(f"\n===== {name} RESULTS =====")
    log(f"Overall ACC = {acc:.4f}")
    log(f"Overall F1  = {f1:.4f}")
    log(f"Overall AUC = {auc:.4f}")
    log(f"Overall FPR = {fpr:.4f}")
    log(f"Confusion Matrix = TN={tn}, FP={fp}, FN={fn}, TP={tp}")

    log("\n--- Year-wise ---")
    print(yearly.to_string(index=False))


# ============================================================
# MAIN
# ============================================================
log("Loading training dataset…")
train_df = pd.read_csv("final_train_data.csv")

gmm, scaler, gmm_train_df, X_scaled = fit_gmm_and_scaler(train_df)
gmm_train_df, alive_clusters = prune_clusters(gmm_train_df, gmm, X_scaled)
rf_models = train_rf_models(gmm_train_df)
thresholds = compute_thresholds(gmm_train_df)

# Run all 3 test evaluations
run_experiment("final_test_data_1.csv", "TEST SET 1 (1:1)")
run_experiment("final_test_data_2.csv", "TEST SET 2 (9:1, DREBIN)")
run_experiment("final_test_data_3.csv", "TEST SET 3 (9:1, AndroZoo)")


In [2]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

def log(s):
    print(s, flush=True)

def load_xy(df):
    X = df.drop(columns=["label", "year"], errors="ignore")
    X = X.select_dtypes(include=[np.number])
    y = df["label"]
    return X, y

# ============================================================
# Load datasets
# ============================================================

train_df = pd.read_csv("final_train_data.csv")

test_files = {
    "Test1_1to1": "final_test_data_1.csv",
    "Test2_9to1_DREBIN": "final_test_data_2.csv",
    "Test3_9to1_AndroZoo": "final_test_data_3.csv",
}

# ============================================================
# Train RF
# ============================================================

X_train, y_train = load_xy(train_df)

log(f"Train feature dim = {X_train.shape[1]}")

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)

rf.fit(X_train, y_train)
log("[OK] RF trained")

# ============================================================
# Evaluation
# ============================================================

years = [2018, 2019, 2020, 2021, 2022, 2023]
results = []

for name, path in test_files.items():
    log(f"\n===== {name} =====")

    test_df = pd.read_csv(path)
    X_test, y_test = load_xy(test_df)

    y_pred = rf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)

    log(f"[Overall] ACC={acc:.4f}, F1={f1:.4f}")

    results.append([name, "Overall", acc, f1])

    for y in years:
        sub = test_df[test_df["year"] == y]
        if len(sub) == 0:
            continue

        X_y, y_y = load_xy(sub)
        y_p = rf.predict(X_y)

        acc_y = accuracy_score(y_y, y_p)
        f1_y  = f1_score(y_y, y_p)

        log(f"[{y}] ACC={acc_y:.4f}, F1={f1_y:.4f}")
        results.append([name, y, acc_y, f1_y])

# ============================================================
# Save
# ============================================================

res_df = pd.DataFrame(
    results,
    columns=["Dataset", "Year", "Accuracy", "F1-score"]
)

res_df.to_csv("rf_baseline_yearwise_results.csv", index=False)
log("\nSaved rf_baseline_yearwise_results.csv")


/tmp/ipykernel_4831/3989689835.py:20: DtypeWarning: Columns (0,1851) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv("final_train_data.csv")


Train feature dim = 1848
[OK] RF trained

===== Test1_1to1 =====


/tmp/ipykernel_4831/3989689835.py:55: DtypeWarning: Columns (0,1851) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv(path)


[Overall] ACC=0.5938, F1=0.7104
[2018] ACC=0.9656, F1=0.9635
[2019] ACC=0.5160, F1=0.6543
[2020] ACC=0.4737, F1=0.6343
[2021] ACC=0.4707, F1=0.6334
[2022] ACC=0.4685, F1=0.6331
[2023] ACC=0.4628, F1=0.6285

===== Test2_9to1_DREBIN =====


/tmp/ipykernel_4831/3989689835.py:55: DtypeWarning: Columns (0,1851) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv(path)


[Overall] ACC=0.2718, F1=0.2154
[2018] ACC=0.9483, F1=0.0000
[2019] ACC=0.1073, F1=0.0000
[2020] ACC=0.0317, F1=0.0000
[2021] ACC=0.0250, F1=0.0000
[2022] ACC=0.0183, F1=0.0000
[2023] ACC=0.0153, F1=0.0000

===== Test3_9to1_AndroZoo =====


/tmp/ipykernel_4831/3989689835.py:55: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv(path)


[Overall] ACC=0.2716, F1=0.2148
[2018] ACC=0.9520, F1=0.8039
[2019] ACC=0.1965, F1=0.1992
[2020] ACC=0.1284, F1=0.1865
[2021] ACC=0.1221, F1=0.1850
[2022] ACC=0.1164, F1=0.1844
[2023] ACC=0.1140, F1=0.1844

Saved rf_baseline_yearwise_results.csv
